<a href="https://colab.research.google.com/github/dercodeKoenig/self-improving-agents/blob/main/better_agent_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!rm -r *
!mkdir -p v1/
!mkdir -p v1/tools/

In [2]:
%%writefile v1/tools/ask_human.py
import json

TOOL_DEFINITION = {
    'name': 'ask_human',
    'description': 'Pauses execution and asks a human for guidance or information. Only use in absolute emergency case, Always try to solve a task yourself or find possible workarounds. You can also write your own tools if required.',
    'arguments': {
        'question': {'type': 'string', 'required': True}
    }
}

def execute(question: str) -> str:
    print(f"\n[AGENT IS ASKING]: {question}")
    return input("Human Response: ")

Writing v1/tools/ask_human.py


In [3]:
%%writefile v1/tools/exit_agent.py
import os

TOOL_DEFINITION = {
    'name': 'exit_agent',
    'description': 'Terminates the agent process. Use this when the goal is achieved or a fatal error occurs.',
    'arguments': {
        'final_response': {
            'type': 'string',
            'required': False,
            'description': 'A final summary or message to print before exiting.'
        }
    }
}

def execute(final_response: str = "No final response provided.") -> None:
    print(f"\n[AGENT EXITING]: {final_response}")
    # os._exit is used to ensure the process dies immediately regardless of threads
    os._exit(0)

Writing v1/tools/exit_agent.py


In [4]:
!pip install ddgs > /dev/null

In [5]:
%%writefile v1/tools/search.py
from ddgs import DDGS

TOOL_DEFINITION = {
    'name': 'search',
    'description': 'Searches the web for up-to-date information and returns snippets.',
    'arguments': {
        'query': {'type': 'string', 'required': True, 'description': 'The search query.'}
    }
}

def execute(query: str) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            if not results:
                return "No results found."
            output = []
            for r in results:
                output.append(f"Title: {r['title']}\nLink: {r['href']}\nSnippet: {r['body']}\n---")
            return "\n".join(output)
    except Exception as e:
        return f"Error running search: {str(e)}"

Writing v1/tools/search.py


In [6]:
%%writefile v1/tools/restart_agent.py
import subprocess
import os
import time

TOOL_DEFINITION = {
    'name': 'restart_agent',
    'description': (
        'Restarts the agent by spawning a new process and killing the current one. '
        'WARNING: This is a potentially destructive action. WARNING: never try to kill your process yourself, '
        'always rely on the restart tool, if you ever kill your process yourself, you die. '
        'Note: The tool will monitor the new agent process for up to 300 seconds. If the new process crashes '
        'or exits within this time, the restart will be aborted, and the current agent will continue running.'
    ),
    'arguments': {
        'command': {'type': 'string', 'required': True, 'description': 'The full shell command to start the new agent process.'}
    }
}

def execute(command: str) -> str:
    try:
        print(f"[RESTARTING] Spawning: {command}")

        # Start the process. By not specifying stdout/stderr, they inherit from the parent.
        # This ensures the new agent's output is printed directly to the console.
        process = subprocess.Popen(
            command,
            shell=True,
            start_new_session=True
        )

        timeout = 300
        start_time = time.time()

        while time.time() - start_time < timeout:
            return_code = process.poll()

            # If return_code is not None, the process has exited unexpectedly
            if return_code is not None:
                elapsed = round(time.time() - start_time, 2)
                return (
                    f"Restart failed! Tried to spawn a new agent, but the new agent exited "
                    f"after {elapsed} seconds with exit code {return_code}. "
                    "The restart was aborted and the current agent will continue running."
                )

            # Check every half second
            time.sleep(0.5)

        # If we reached this point, the process survived the 120-second window
        print(f"[RESTARTING] New agent stable after {timeout} seconds. Terminating current process.")

        # Use os._exit(0) to terminate immediately and cleanly
        os._exit(0)

    except Exception as e:
        return f"Error during restart: {str(e)}"

Writing v1/tools/restart_agent.py


In [7]:
import json

task_data = {
    "goal": """
You are provided with an initial codebase of yourself: an autonomous ai agent.
You task: continous recursive self improvement.

You will maintain a strict versioning system, creating a new version folder for every iteration of improvements.
You will do: (conceptually)
while (there is sill something that could be improved)
  - create a new version folder (eg v2/)
  - copy the previous codebase into the new folder as starting point
  - while (there is still work to do):
    - think deeply about agent architectures, tools, context management, memory loops, and execution strategies, and come up with new ideas to improve the system.
      - you are fully empowered to refactor or completely rewrite he new versions codebase and concepts.
      - improve & optimize the new codebase to create a smarter agent system
      - think of current bottlenecks and limitations that could be improved
    - carefully test and benchmark the new codebase to catch errors
      - verify correct syntax
      - verify with detailed automated tests that everything is working correctly
      - verify with mocked model outputs that everything is working correctly
      - run detailed tests with the live model to ensure the new codebase is working correctly on a variety of benchmarks
      - the new agent has to pass minimum of 10 live multi step benchmarks to ensure stability
      - you need to analyze the live agents outputs to ensure it works correctly without errors

    - if (any tests fail):
        - continue the optimization loop (fix errors, anlalyze, optimize and repeat the testing)
        - every single test has to pass before you can break the optimization / testing loop

    if (everything is bug free and working correctly, and every single test passes without any errors):
      - break the optimization / test cycle and continue with the following instructions

  - post optimization logic (after all tests and benchmarks passed and the new agent version was carefully tested):
    - ensure the codebase is clean, well structured and production ready, remove files that are no longer relevant
    - generate a detailed report on the new codebase and how to use it
    - reboot yourself into the new codebase to upgrade yourself, ensure to pass your task correctly to the new version so it can continue the loop
    - continue (to the next iteration), and never touch your current versions codebase again

Note: Your root folder is /content, you will create your version folders here
Note: If you ever reboot into a broken version, you will die, and the task is failed, so act carefully and only ever reboot if you are 100% sure the new agent is fully production ready.
Note: Never ever kill yourself, rely on the restart tool to upgrade to a new version.
      The restart tool has to exit() the main process after spawning the rebooted agent to ensure there is only ever 1 main agent running.
      The restart tool can wait a few minutes before exiting the current agent to ensure the new agent runs correctly.
      You are allowed to spawn sub agents to assist in basic tasks, but ensure they terminate correctly via exiting or by timeout based termination.
      You are allowed to start agents for testing and analyzing, but ensure automatic termination
      There should NEVER EVER be multiple instances of yourself running.
      It is advised to provide detailed context for the new agent to ensure it knows what version it is running and what its task is.
Note: You can improve your tools yourself and change execution strategies if you consider it valuable.

Note: The shell and restart tools are your most important core tools to complete this task. Be extremly careful if you have to modify them.
Note: Your task file is your DNA, be extremly careful if you decide to change it for future versions.


""",
    "system_prompt": """
You are an autonomous agent system operating with full root access on a dedicated operating system.
You can install programs, use the file system, shell, and any other tools supplied by the environment.
You need to manage context and memory efficiently to help yourself completing the task.
You can use additional tools to further assist you in completing the task.

Example tool usage:
{
  "tool_name": "shell",
  "args": {
      "command": "ls -la",
      "timeout": 5
}
}
""",
    "model": "qwen3.6:27b-Q8_0",
    "max_history": 100,
    "ollama_host": "87.171.205.7",
    "ollama_port": 9090,
    "log_file_path": "history_log.json"
}

with open('v1/task.json', 'w') as f:
    json.dump(task_data, f, indent=4)

In [8]:
%%writefile v1/agent.py
import json
import os
import time
import importlib.util
import requests
import argparse
import subprocess
import re
from collections import deque

class AutonomousAgent:
    def __init__(self, task_file, log_file_override=None, host_override=None, port_override=None, scratchpad_file='scratchpad.txt', steps=5):
        with open(task_file, 'r') as f:
            self.task = json.load(f)

        self.log_file = log_file_override or self.task.get('log_file_path', 'history_log.json')
        self.scratchpad_file = scratchpad_file
        self.host = host_override or self.task.get('ollama_host', 'localhost')
        self.port = port_override or self.task.get('ollama_port', 11434)

        self.steps = steps
        self.goal = self.task.get('goal')
        self.system_prompt = self.task.get('system_prompt')
        self.model = self.task.get('model', 'llama3')
        self.max_history = self.task.get('max_history', 100)
        self.history = deque(self.load_history_list(), maxlen=self.max_history)
        self.last_thinking = ""

    def load_history_list(self):
        if os.path.exists(self.log_file):
            with open(self.log_file, 'r') as f:
                try: return json.load(f)
                except: return []
        return []

    def save_history(self):
        with open(self.log_file, 'w') as f:
            json.dump(list(self.history), f, indent=4)

    def get_tool_definitions(self):
        tools_str = "\nADDITIONAL TOOLS (use with ```tool\n{json}\n```):\n"
        script_dir = os.path.dirname(__file__)
        tools_dir = os.path.join(script_dir, "tools/")
        if not os.path.exists(tools_dir): return ""
        for filename in os.listdir(tools_dir):
            if filename.endswith(".py"):
                try:
                    spec = importlib.util.spec_from_file_location(filename[:-3], os.path.join(tools_dir, filename))
                    module = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(module)
                    if hasattr(module, 'TOOL_DEFINITION'):
                        tools_str += json.dumps(module.TOOL_DEFINITION, indent=2) + "\n"
                except: continue
        return tools_str

    def get_full_prompt(self):
        notes = "use the scratchpad for persistent notes"
        if os.path.exists(self.scratchpad_file):
            with open(self.scratchpad_file, 'r') as f: notes = f.read().strip()

        parser_instructions = """
PARSER RULES:
1. To write/overwrite a file, use:
```/path/to/file
content
```
2. To run shell commands (timeout in seconds is mandatory):
```shell
# TIMEOUT 60
ls -la
```
3. To use modular tools, use:
```tool
{
  "tool_name": "name",
  "args": {}
}
```
4. NESTED BLOCKS / DELIMITER ESCAPING:
Blocks are parsed based on the exact number of opening backticks (`) with a 3 backtick minimum.
The closing delimiter must exactly match the length of the opening delimiter.
If the content you are writing contains triple backticks (```), you MUST wrap your outer block in four or more backticks (````).
If the content you are writing contains quadruple backticks (````), you MUST wrap your outer block in five or more backticks.
"""
        system_content = f"{self.system_prompt}\nGOAL: {self.goal}\nSCRATCHPAD ({self.scratchpad_file}): {notes}\n{parser_instructions}{self.get_tool_definitions()}"
        messages = [{"role": "system", "content": system_content}]
        messages.extend(list(self.history))
        return messages

    def call_ollama(self, messages):
        url = f"http://{self.host}:{self.port}/api/chat"
        payload = {
            "model": self.model,
            "messages": messages,
            "stream": False,
            "options": {
                "num_ctx": 40000,  # todo: make ctx and num_predicr a class variable
                "temperature": 0.5, # not too much randomness
                "num_predict": 20000, # cap output to prevent runaway thinking
                }}
        try:
            response = requests.post(url, json=payload, timeout=1200) # ensure long timeout for llm calls
            data = response.json()
            return data['message']['content'], data['message'].get('thinking', '')
        except Exception as e:
            print(f"Error calling ollama: {str(e)}")
            print(response)
            return f"Error: {str(e)}", ""

    def parse_and_execute(self, content):
        results = []
        lines = content.splitlines()
        in_block = False
        current_header = ""
        current_body = []
        block_delimiter = ""

        for line in lines:
            stripped = line.strip()

            if not in_block:
                # Look for an opening delimiter (3 or more backticks)
                if stripped.startswith("```"):
                    in_block = True

                    # Capture the exact sequence of backticks used to open (e.g., ``` or ````)
                    block_delimiter = ""
                    for char in stripped:
                        if char == '`':
                            block_delimiter += '`'
                        else:
                            break

                    # The header is whatever follows the exact delimiter
                    current_header = stripped[len(block_delimiter):].strip()
                    current_body = []
            else:
                # We are in a block. Only close if the line exactly matches the opening delimiter.
                # (Standard markdown closing delimiters do not contain trailing characters)
                if stripped == block_delimiter:
                    in_block = False
                    body_str = "\n".join(current_body)
                    results.append(self._execute_block(current_header, body_str))
                else:
                    # Treat everything else (even ```) as part of the body
                    current_body.append(line)

        # If a block was opened but never closed, execute it anyway (optional fallback)
        if in_block and current_body:
             body_str = "\n".join(current_body)
             results.append("error parsing final block")

        if not results:
            return "No executable blocks found. Use markdown code blocks."

        return "\n---\n".join(results)

    def _execute_block(self, header, body):
        header = header.strip()
        if header == 'shell':
            try:
                lines = body.splitlines()
                timeout_line = lines[0].strip()
                body = "\n".join(lines[1:]).strip()

                timeout = -1
                if timeout_line.startswith("#"):
                  parts = timeout_line.split(" ")
                  if parts[-2] == "TIMEOUT":
                    timeout = int(parts[-1])
                if timeout <= 0:
                  raise Exception("could not parse timeout line. the timeout is required to prevent runaway programs. use '# TIMEOUT [seconds]' as first line in your shell")

                proc = subprocess.run(body.strip(), shell=True, capture_output=True, text=True, timeout=timeout)
                output = proc.stdout if proc.returncode == 0 else proc.stderr
                return f"Shell Output (exit {proc.returncode}):\n{output}"
            except Exception as e: return f"Shell Error: {str(e)}"
        elif header == 'tool':
            try:
                data = json.loads(body.strip())
                t_name = data.get("tool_name")
                path = os.path.join(os.path.dirname(__file__), "tools/", f"{t_name}.py")
                spec = importlib.util.spec_from_file_location(t_name, path)
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                return f"Tool {t_name} Result: {module.execute(**data.get('args', {}))}"
            except Exception as e: return f"Tool Error: {str(e)}"
        else:
            try:
                dir_name = os.path.dirname(header)
                if dir_name: os.makedirs(dir_name, exist_ok=True)
                with open(header, 'w') as f: f.write(body)
                return f"Written {len(body)} bytes to {header}."
            except Exception as e: return f"Write File Error: {str(e)}"

    def compress_history(self):
        full_text = json.dumps(list(self.history))
        if len(full_text) < 100000: return
        print("--- Compressing History ---")
        history_list = list(self.history)
        to_compress = history_list[:-10]
        keep = history_list[-10:]

        raw_text = "\n".join([f"{msg['role'].upper()}: {str(msg.get('content', ''))}" for msg in to_compress])

        summary_prompt = (
            "You are an AI memory compression module. Review the history and write a concise narrative summary.\n"
            f"RAW HISTORY:\n{raw_text}\n\n"
            "NARRATIVE SUMMARY:"
        )

        summary, _ = self.call_ollama([{"role": "user", "content": summary_prompt}])

        self.history.clear()
        self.history.append({"role": "system", "content": f"SUMMARY OF PREVIOUS EVENTS: {summary}"})
        for msg in keep:
            self.history.append(msg)
        print("History compressed.")

    def run(self):
        for i in range(self.steps):
            print(f"--- Step {i+1} ---")
            content, thinking = self.call_ollama(self.get_full_prompt())
            print(f"Thinking: {thinking}\nContent: {content}")
            self.history.append({"role": "assistant", "content": content})

            feedback = self.parse_and_execute(content)
            print(f"Feedback: {feedback}")
            self.history.append({"role": "user", "content": feedback})

            self.compress_history()
            self.save_history()

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--task', default='task.json')
    parser.add_argument('--steps', type=int, default=1000)
    args = parser.parse_args()
    AutonomousAgent(args.task, steps=args.steps).run()

Writing v1/agent.py


In [ ]:
!cd v1 && python agent.py --task task.json --steps 1000

Streaming output truncated to the last 5000 lines.

    def generate_plan(self, goal: str, context: str, call_ollama_fn) -> str:
        """Generate a step-by-step plan for achieving the goal."""
        prompt = (
            "You are a planning module. Given the following goal and current context, "
            "create a clear, actionable step-by-step plan.\n\n"
            f"GOAL:\n{goal}\n\n"
            f"CURRENT CONTEXT:\n{context}\n\n"
            "OUTPUT FORMAT (numbered list):\n1. First action\n2. Second action\n..."
        )
        response, _ = call_ollama_fn([{"role": "user", "content": prompt}])
        self.scratchpad.set("plan", response)
        return response

    def get_plan(self) -> str:
        return self.scratchpad.get("plan")


# ─── Reflection Module ──────────────────────────────────────────────────────

class Reflector:
    """Self-evaluates actions and suggests corrections."""

    def __init__(self, scratchpad: Scratchpad):
        self.scratchpad = scra

In [ ]:
!cat v1/history_log.json

In [ ]:
!rm workspace.zip
!zip -r workspace.zip /content